# Initialization

In [0]:
import pyspark.sql.functions as F
from pyspark.sql.types import StringType
from pyspark.sql.functions import trim, col

# Read Bronze Table

In [0]:
df = spark.table("databricks_medallion_pipeline.bronze.crm_sales_details")
df.limit(15).display()

# Silver Transformations

##Trimming

In [0]:
for field in df.schema.fields:
    if isinstance(field.dataType, StringType):
        df.withColumn(field.name, trim(col(field.name)))    

##Cleaning Dates

In [0]:
df = (
    df
    .withColumn(
        "sls_order_dt",
        F.when(
            (col("sls_order_dt") == 0) | (F.length(col("sls_order_dt")) != 8),
            None
        ).otherwise(F.to_date(col("sls_order_dt").cast("string"), "yyyyMMdd"))
    )
    .withColumn(
        "sls_ship_dt",
        F.when(
            (col("sls_ship_dt") == 0) | (F.length(col("sls_ship_dt")) != 8),
            None
        ).otherwise(F.to_date(col("sls_ship_dt").cast("string"), "yyyyMMdd"))
    )
    .withColumn(
        "sls_due_dt",
        F.when(
            (col("sls_due_dt") == 0) | (F.length(col("sls_due_dt")) != 8),
            None
        ).otherwise(F.to_date(col("sls_due_dt").cast("string"), "yyyyMMdd"))
    )
)

## Sales and Price Corrections

In [0]:
df = (
    df
    .withColumn(
        "sls_price",
        F.when(
            (col("sls_price").isNull()) | (col("sls_price") <= 0),
            F.when(
                col("sls_quantity") != 0,
                col("sls_sales") / col("sls_quantity")
            ).otherwise(None)
        ).otherwise(col("sls_price"))
    )
)

# Renaming Columns

In [0]:
RENAME_DICT = {
    "sls_ord_num": "order_number",
    "sls_prd_key": "product_number",
    "sls_cust_id": "customer_id",
    "sls_order_dt": "order_date",
    "sls_ship_dt": "ship_date",
    "sls_due_dt": "due_date",
    "sls_sales": "sales_amount",
    "sls_quantity": "quantity",
    "sls_price": "price"
}

for old_name, new_name in RENAME_DICT.items():
    df = df.withColumnRenamed(old_name, new_name)

#Verifying Dataframe and Saving

## Printing rows for verification

In [0]:
df.limit(15).display()

##Writing Silver Table

In [0]:
df.write.mode("overwrite").saveAsTable("databricks_medallion_pipeline.silver.crm_sales")

#Verifying Saved Table

In [0]:
df = spark.sql("SELECT * FROM databricks_medallion_pipeline.silver.crm_sales")
df.display()